# ⚙️ Fase 1 — Preprocesamiento y Extracción HOG
**Pipeline:** Imagen RAW → Recorte BBox → Resize 64×64 → Escala de Grises → CLAHE → HOG → Vector 1D

In [ ]:
import cv2, numpy as np, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.feature import hog
from skimage import exposure
from tqdm import tqdm
print('✅ Librerías OK')

In [ ]:
PROJECT_ROOT   = Path('..').resolve()
DATA_RAW       = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMG_SIZE = (64, 64)
print('Config OK — IMG_SIZE:', IMG_SIZE)

In [ ]:
def preprocesar_imagen(img_path, bbox=None):
    img = cv2.imread(str(img_path))
    if img is None: return None, None
    if bbox:
        x,y,w,h = bbox
        img = img[y:y+h, x:x+w]
    img = cv2.resize(img, IMG_SIZE)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    vec, hog_img = hog(gray, orientations=9, pixels_per_cell=(8,8),
                       cells_per_block=(2,2), visualize=True, feature_vector=True)
    return vec, hog_img
print('✅ Función definida')

In [ ]:
# Visualizar pipeline en imagen de ejemplo
sample = list((DATA_RAW / 'images' / 'train').glob('*.jpg'))[:1]
if sample:
    vec, hog_img = preprocesar_imagen(sample[0])
    img_orig = cv2.cvtColor(cv2.resize(cv2.imread(str(sample[0])), IMG_SIZE), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1,3,figsize=(15,5))
    axes[0].imshow(img_orig); axes[0].set_title('1. Original'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(img_orig, cv2.COLOR_RGB2GRAY), cmap='gray'); axes[1].set_title('2. Gris+CLAHE'); axes[1].axis('off')
    axes[2].imshow(exposure.rescale_intensity(hog_img,(0,10)), cmap='inferno'); axes[2].set_title(f'3. HOG ({len(vec)} features)'); axes[2].axis('off')
    plt.suptitle('Pipeline HOG', fontsize=14, fontweight='bold'); plt.tight_layout(); plt.show()
    print(f'✅ Vector HOG: {len(vec)} características')
else:
    print('⚠️  Coloca el dataset en data/raw/')

In [ ]:
def procesar_split(split):
    img_dir = DATA_RAW/'images'/split; label_dir = DATA_RAW/'labels'/split
    X,y = [],[]
    imgs = list(img_dir.glob('*.jpg'))+list(img_dir.glob('*.png'))
    print(f'\n🔄 {split}: {len(imgs)} imágenes')
    for ip in tqdm(imgs):
        img = cv2.imread(str(ip))
        if img is None: continue
        h_img,w_img = img.shape[:2]
        lp = label_dir/(ip.stem+'.txt')
        if lp.exists():
            lines = open(lp).readlines()
            if lines:
                for ln in lines:
                    pts = ln.strip().split()
                    if len(pts)==5:
                        _,xc,yc,bw,bh = [float(p) for p in pts]
                        bbox=(max(0,int((xc-bw/2)*w_img)),max(0,int((yc-bh/2)*h_img)),int(bw*w_img),int(bh*h_img))
                        v,_ = preprocesar_imagen(ip, bbox)
                        if v is not None: X.append(v); y.append(1)
            else:
                v,_ = preprocesar_imagen(ip)
                if v is not None: X.append(v); y.append(0)
        else:
            v,_ = preprocesar_imagen(ip)
            if v is not None: X.append(v); y.append(0)
    X,y = np.array(X),np.array(y)
    print(f'   Shape:{X.shape} | Baches:{y.sum()} | Normal:{(y==0).sum()}')
    return X,y

X_train,y_train = procesar_split('train')
X_val,y_val     = procesar_split('val')
X_test,y_test   = procesar_split('test')

In [ ]:
np.save(DATA_PROCESSED/'X_train.npy',X_train); np.save(DATA_PROCESSED/'y_train.npy',y_train)
np.save(DATA_PROCESSED/'X_val.npy',X_val);     np.save(DATA_PROCESSED/'y_val.npy',y_val)
np.save(DATA_PROCESSED/'X_test.npy',X_test);   np.save(DATA_PROCESSED/'y_test.npy',y_test)
print('✅ Features guardadas en data/processed/')